# Week 6 – Werkcollege 9: Visual Maandag — Information Architecture & Progressive Disclosure

Geen nieuwe inleveropdracht vandaag. Je herontwerpt de kaart uit Week 5: van "alles tegelijk zichtbaar" naar twee lagen — een simpel eerste beeld, en detail dat pas verschijnt als iemand erom vraagt.

## Tijdsindicatie

| Moment | Duur | Onderdeel |
|---|---|---|
| Wat is progressive disclosure? | 10 min | Een overvolle kaart herkennen |
| Data verzamelen | 15 min | Locaties + verdiepende stats per bot |
| Level 1: simpele kaart | 15 min | Alleen kleur en omvang, verder niets |
| Level 2: detail-popup | 25 min | Sparkline + statistieken, pas zichtbaar na klik |
| Layout-discipline | 10 min | Wat laat je expres weg? |
| Zoom-afhankelijke clustering (bonus) | 10 min | Een derde laag: clusteren bij uitzoomen |
| Peer-vergelijking | 10 min | Feedback van een klasgenoot |


## Deel 1 — Wat is progressive disclosure? (10 min)

Progressive disclosure betekent: laat eerst alleen het belangrijkste zien, en verberg de rest achter een interactie (een klik, een hover, een zoom). Het doel is niet om informatie weg te stoppen, maar om te voorkomen dat de lezer alles tegelijk over zich heen krijgt.

Hieronder staat een kaart die dat níet doet: elke marker toont zijn hele statistiek-blok permanent, in de tooltip.


In [ ]:
import folium

fictieve_bots = [
    {"naam": "Bot_A", "lat": 52.37, "lon": 4.90, "eindstand": 1400, "call_pct": 40, "raise_pct": 35, "fold_pct": 25, "aantal_handen": 250},
    {"naam": "Bot_B", "lat": 51.92, "lon": 4.48, "eindstand": 700, "call_pct": 20, "raise_pct": 15, "fold_pct": 65, "aantal_handen": 250},
]

overvolle_kaart = folium.Map(location=[52.1, 5.1], zoom_start=7)
for bot in fictieve_bots:
    tekst = (
        f"{bot['naam']} | eindstand: {bot['eindstand']} | "
        f"call: {bot['call_pct']}% raise: {bot['raise_pct']}% fold: {bot['fold_pct']}% | "
        f"handen: {bot['aantal_handen']}"
    )
    folium.Marker([bot["lat"], bot["lon"]], tooltip=tekst).add_to(overvolle_kaart)

overvolle_kaart


🤔 Beweeg over beide markers zonder te klikken. Welke informatie had hier prima kunnen wachten tot na een klik?


## Deel 2 — Data verzamelen (15 min)

Haal de locaties en het volledige toernooi van Week 5 op, en bereken per bot wat je nodig hebt voor straks: hoe vaak hij call/raise/fold koos, en over hoeveel handen dat ging.


In [ ]:
import requests
import pandas as pd

API_URL = "http://localhost:8000"
STUDENT_ID = "vul_hier_je_student_id_in"
TOKEN = "vul_hier_je_token_in"

locaties = requests.get(
    f"{API_URL}/locaties/5",
    params={"student_id": STUDENT_ID},
    headers={"Authorization": f"Bearer {TOKEN}"},
).json()

toernooi = pd.DataFrame(requests.get(
    f"{API_URL}/toernooi/5",
    params={"student_id": STUDENT_ID},
    headers={"Authorization": f"Bearer {TOKEN}"},
).json()["hand_log"])

len(locaties), toernooi.shape


In [ ]:
actie_verdeling = (
    toernooi.groupby("bot_naam")["actie"]
    .value_counts(normalize=True)
    .unstack(fill_value=0.0) * 100
)
aantal_handen_per_bot = toernooi.groupby("bot_naam").size()
actie_verdeling.head()


🤔 `value_counts(normalize=True)` geeft percentages in plaats van aantallen. Waarom is een percentage hier een eerlijkere vergelijking tussen bots dan een los aantal?


## Deel 3 — Level 1: simpele kaart (15 min)

Dit is wat een lezer als eerste ziet. Kleur op winst/verlies, omvang op hoe groot het verschil met de startstack (1000) is — en verder niets zichtbaars. Geen tooltip met cijfers, geen popup nog.


In [ ]:
kaart = folium.Map(location=[52.1, 5.1], zoom_start=7)

for bot in locaties:
    eindstand = bot["eindstand"] if bot["eindstand"] is not None else 1000
    kleur = "green" if eindstand >= 1000 else "red"
    omvang = max(5, min(20, abs(eindstand - 1000) / 50))

    folium.CircleMarker(
        location=[bot["lat"], bot["lon"]],
        radius=omvang,
        color=kleur,
        fill=True,
        fill_color=kleur,
        fill_opacity=0.7,
    ).add_to(kaart)

kaart


🎨 Er staat nu geen tekst meer op de kaart. Kun je zonder te klikken toch al zien wie er ongeveer het meest gewonnen heeft? Waaraan zie je dat?


## Deel 4 — Level 2: detail-popup (25 min)

Alles wat in Deel 1 nog permanent zichtbaar was, verhuist nu naar een popup: een mini-grafiekje (sparkline) van het stackverloop, plus de call/raise/fold-percentages. Dat verschijnt pas na een klik.

Een sparkline is een kleine, aslabel-loze lijngrafiek — puur bedoeld om een patroon in één oogopslag te laten zien, niet om exacte waarden af te lezen.


In [ ]:
import io
import base64
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def maak_sparkline(bot_naam, toernooi):
    eigen_rijen = toernooi[toernooi["bot_naam"] == bot_naam]
    if eigen_rijen.empty:
        return None
    eerste_tafel, eerste_sim = eigen_rijen[["tafel", "simulatie"]].iloc[0]
    reeks = eigen_rijen[
        (eigen_rijen["tafel"] == eerste_tafel) & (eigen_rijen["simulatie"] == eerste_sim)
    ].sort_values("hand_nummer")

    fig, ax = plt.subplots(figsize=(2.4, 0.7))
    ax.plot(reeks["hand_nummer"], reeks["stack"], color="#2E86AB", linewidth=1.5)
    ax.axis("off")
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=100, bbox_inches="tight", transparent=True)
    plt.close(fig)
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("utf-8")

test_sparkline = maak_sparkline(locaties[0]["student_id"], toernooi)
test_sparkline is not None


Bouw nu de popup: sparkline-afbeelding + de 3 percentages + aantal handen. `folium.IFrame` zorgt ervoor dat HTML met een `<img>`-tag goed in een popup terechtkomt.


In [ ]:
def maak_popup_html(bot, actie_verdeling, aantal_handen_per_bot, toernooi):
    naam = bot["student_id"]
    sparkline = maak_sparkline(naam, toernooi)
    afbeelding_tag = (
        f'<img src="data:image/png;base64,{sparkline}" width="150"><br>' if sparkline else ""
    )

    if naam in actie_verdeling.index:
        percentages = actie_verdeling.loc[naam]
        percentage_tekst = " · ".join(
            f"{actie}: {percentages.get(actie, 0):.0f}%" for actie in ["call", "raise", "fold"]
        )
    else:
        percentage_tekst = "nog geen data"

    return f"""
    <div style="font-family: sans-serif; font-size: 12px;">
      <b>{naam}</b> ({bot['plaatsnaam']})<br>
      Eindstand: {bot['eindstand']}<br>
      {afbeelding_tag}
      {percentage_tekst}<br>
      Handen gespeeld: {aantal_handen_per_bot.get(naam, '?')}
    </div>
    """

kaart = folium.Map(location=[52.1, 5.1], zoom_start=7)

for bot in locaties:
    eindstand = bot["eindstand"] if bot["eindstand"] is not None else 1000
    kleur = "green" if eindstand >= 1000 else "red"
    omvang = max(5, min(20, abs(eindstand - 1000) / 50))

    popup_html = maak_popup_html(bot, actie_verdeling, aantal_handen_per_bot, toernooi)
    popup = folium.Popup(folium.IFrame(popup_html, width=200, height=180), max_width=250)

    folium.CircleMarker(
        location=[bot["lat"], bot["lon"]],
        radius=omvang, color=kleur, fill=True, fill_color=kleur, fill_opacity=0.7,
        popup=popup,
    ).add_to(kaart)

kaart


🎨 Klik op een paar markers. Vergelijk dit met Deel 1: is de informatie nu weg, of alleen uitgesteld?


## Deel 5 — Layout-discipline (10 min)

De verleiding bij progressive disclosure is om de popup zelf weer vol te proppen, omdat er nu "toch ruimte" is. Dat is dezelfde fout als in Deel 1, één laag dieper.

Bedenk 1 statistiek die je zou kúnnen toevoegen aan de popup (bv. langste win-streak, aantal keer gebluft, gemiddelde inzet) en beargumenteer waarom je 'm juist NIET toevoegt.


In [ ]:
# jouw statistiek + argument waarom je 'm weglaat



🤔 Is er een verschil tussen "ik laat dit weg omdat het niet past" en "ik laat dit weg omdat het niet relevant is"? Welke van de twee gold voor jouw statistiek?


## Deel 6 — Bonus: zoom-afhankelijke clustering (10 min)

Een derde vorm van progressive disclosure: bij een grote klas (of een heel land vol bots) wil je niet alle markers tegelijk zien zodra je uitzoomt. `MarkerCluster` groepeert nabije markers automatisch tot één cijfer, en splitst ze weer op zodra je inzoomt.


In [ ]:
from folium.plugins import MarkerCluster

kaart_geclusterd = folium.Map(location=[52.1, 5.1], zoom_start=7)
cluster = MarkerCluster().add_to(kaart_geclusterd)

for bot in locaties:
    eindstand = bot["eindstand"] if bot["eindstand"] is not None else 1000
    kleur = "green" if eindstand >= 1000 else "red"
    folium.CircleMarker(
        location=[bot["lat"], bot["lon"]],
        radius=8, color=kleur, fill=True, fill_color=kleur,
        tooltip=bot["student_id"],
    ).add_to(cluster)

kaart_geclusterd


🤔 Clustering verbergt tijdelijk hoeveel bots er in een gebied zitten, achter één getal. Is dat nog steeds progressive disclosure, of is het iets anders? Waarin verschilt het van de popup uit Deel 4?


## Deel 7 — Peer-vergelijking (10 min)

Wissel je kaart uit met een klasgenoot. Laat diegene 3 seconden naar Level 1 kijken (zonder te klikken) en vraag wat ze zien. Klik daarna pas door naar Level 2.


In [ ]:
# wat zag je klasgenoot bij Level 1, voordat er geklikt werd?



---

## Reflectievragen

🎨 Vergelijk de kaart uit Deel 1 met die uit Deel 4. Welke informatie verplaatste je, en welke liet je volledig weg?

🤔 Progressive disclosure voegt een extra handeling toe (een klik) om bij informatie te komen. Wanneer is die extra moeite het waard, en wanneer niet?

💡 Een sparkline heeft bewust geen assen of labels. Wat zou je patroon-herkenning kapotmaken als je die er alsnog aan toevoegde?

🤔 In Deel 5 heb je iets bewust weggelaten. Zou je die keuze anders maken als deze kaart voor de docent was in plaats van voor klasgenoten?

🎨 Marker-grootte (Deel 3) en clustering (Deel 6) zijn allebei manieren om met veel data om te gaan zonder alles te tonen. Wanneer kies je voor het één, wanneer voor het ander?

---

### Vooruitblik: Week 7

De Hackathon. Daar komen we later op terug.
